In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader = PyPDFLoader("Submission.pdf")
document =loader.load()
len(document)

7

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size =800,chunk_overlap =200)
splitted_text = splitter.split_documents(document)
len(splitted_text)

14

In [5]:
embedding = OpenAIEmbeddings(model="text-embedding-3-large")
vector_stored = InMemoryVectorStore.from_documents(
    documents=splitted_text,
    embedding=embedding,
    
)

In [9]:
@tool
def retriver_tool(query:str):
    """
        this tool can help you find relevant infromation from the pdf document, and these document may contain information about project.
    """
    docs = vector_stored.similarity_search(query=query)
    context =""

    for doc in docs:
        context = doc.page_content + "\n\n"
    return context

In [10]:
llm = ChatOpenAI(model="gpt-4o")

In [11]:
System_prompt= """
        you are a helpful assistant that answer questions using the retrived context.
        ALWAYS use the 'retrivever_tool' tool for question requiring external knowledge.
"""

In [15]:
agent = create_agent(
    model=llm,
    tools=[retriver_tool],
    system_prompt=System_prompt,
)

In [17]:
query=input("user :")
response = agent.invoke({"messages":[{"role":"user","content":query}]})
response["messages"][-1].content

'The assessment involves a system architecture and workflow for processing video content, which includes the following steps:\n\n1. **Audio Transcription**: The system transcribes the audio content using WhisperX.\n2. **Clip Selection**: It selects the best moments of the video using Gemini 2.5-flash, with NVIDIA NIM Llama 3.3-70b as a fallback option.\n3. **Active Speaker Detection**: The system detects the active speaker in each video clip using TalkNet-ASD.\n4. **Video Rendering**: It renders portrait-mode clips (9:16 format) with burned-in captions and a LUNARTECH.AI watermark using ffmpeg.\n5. **Output Delivery**: Finally, the processed clips are delivered back to the dashboard for preview and download.\n\n### Inngest System\n\nInngest in this context refers to a queue and orchestration system that is part of the overall pipeline, integrating various components:\n\n- **Queue Management**: Inngest is used to manage concurrency to prevent GPU resource contention.\n- It is integrated

In [18]:
print(response["messages"][-1].content)

The assessment involves a system architecture and workflow for processing video content, which includes the following steps:

1. **Audio Transcription**: The system transcribes the audio content using WhisperX.
2. **Clip Selection**: It selects the best moments of the video using Gemini 2.5-flash, with NVIDIA NIM Llama 3.3-70b as a fallback option.
3. **Active Speaker Detection**: The system detects the active speaker in each video clip using TalkNet-ASD.
4. **Video Rendering**: It renders portrait-mode clips (9:16 format) with burned-in captions and a LUNARTECH.AI watermark using ffmpeg.
5. **Output Delivery**: Finally, the processed clips are delivered back to the dashboard for preview and download.

### Inngest System

Inngest in this context refers to a queue and orchestration system that is part of the overall pipeline, integrating various components:

- **Queue Management**: Inngest is used to manage concurrency to prevent GPU resource contention.
- It is integrated with Supabase